In [16]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

import pandas as pd
import joblib

from data import load_and_split
from evaluate import evaluate
from compare import per_user_metrics, paired_difference
from models.popularity import Popularity
from models.random_model import RandomModel
from models.pure_svd import PureSVD
from models.ials import IALS
from models.bpr import BPR
from models.ease import EASE

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
train, val, test = load_and_split('../data/ml-1m/ratings.dat')
print(train['user'].nunique(), train['item'].nunique())
print(len(train), len(val), len(test))

6034 3125
562308 6034 6034


In [8]:
train_val = pd.concat([train, val], ignore_index=True)

factories = {
    'random': lambda: RandomModel(),
    'popularity': lambda: Popularity(),
    'bpr': lambda: BPR(n_factors=64, learning_rate=0.01, n_iter=100),
    'puresvd': lambda: PureSVD(n_factors=64),
    'ials': lambda: IALS(n_factors=64, alpha=1, reg=0.1, n_iter=15),
    'ease': lambda: EASE(reg=500),
}


def run(fit_data):
    rows, recalls, ndcgs = [], {}, {}
    for name, make in factories.items():
        model = make().fit(fit_data)
        recall, ndcg = per_user_metrics(model, fit_data, test, k=10)
        recalls[name], ndcgs[name] = recall, ndcg
        rows.append({
            'model': name,
            'recall': recall.mean(),
            'recall_ci': 1.96 * recall.std(ddof=1) / len(recall) ** 0.5,
            'ndcg': ndcg.mean(),
        })
    return pd.DataFrame(rows).set_index('model'), recalls, ndcgs


table_a, rec_a, ndcg_a = run(train)
table_b, rec_b, ndcg_b = run(train_val)

In [12]:
table_a

,recall,recall_ci,ndcg
model,,,
random,0.003812,0.001555,0.001874
popularity,0.038615,0.004862,0.018972
bpr,0.068114,0.006358,0.034235
puresvd,0.076566,0.006710,0.036800
ials,0.073583,0.006588,0.035805
ease,0.081704,0.006912,0.040893


In [13]:
table_b

,recall,recall_ci,ndcg
model,,,
random,0.003812,0.001555,0.001874
popularity,0.038780,0.004872,0.018999
bpr,0.073583,0.006588,0.036048
puresvd,0.088167,0.007155,0.043188
ials,0.087173,0.007118,0.043508
ease,0.100597,0.007590,0.049324


In [9]:
pairs = [('ease', 'puresvd'), ('ease', 'ials'), ('ials', 'puresvd'), ('puresvd', 'bpr')]
for a, b in pairs:
    print(a, b, paired_difference(rec_b[a], rec_b[b]))

ease puresvd {'mean_a': 0.10059661915810408, 'mean_b': 0.08816705336426914, 'diff': 0.012429565793834935, 'se': 0.0031362690177465457, 'ci95': (0.006282478519051706, 0.018576653068618165), 'z': 3.9631695251594703, 'p_value': 7.396122227698057e-05, 'n_users': 6034}
ease ials {'mean_a': 0.10059661915810408, 'mean_b': 0.08717268810076234, 'diff': 0.01342393105734173, 'se': 0.0029365344559788634, 'ci95': (0.007668323523623158, 0.0191795385910603), 'z': 4.5713514547769885, 'p_value': 4.8458874517818605e-06, 'n_users': 6034}
ials puresvd {'mean_a': 0.08717268810076234, 'mean_b': 0.08816705336426914, 'diff': -0.0009943652635067948, 'se': 0.002610064866857818, 'ci95': (-0.006110092402548118, 0.004121361875534528), 'z': -0.3809733911724127, 'p_value': 0.7032229930623166, 'n_users': 6034}
puresvd bpr {'mean_a': 0.08816705336426914, 'mean_b': 0.07358302949950282, 'diff': 0.014584023864766324, 'se': 0.004075741502919922, 'ci95': (0.006595570519043277, 0.02257247721048937), 'z': 3.578250449474804, 

In [7]:
leak = pd.concat([train, test], ignore_index=True)
leak_model = EASE(reg=1.0).fit(leak)
recall, _ = per_user_metrics(leak_model, train, test, k=10)
print(recall.mean())

0.7699701690420948


#### Метрики в этом проекте считаются по полному каталогу, метрики из литературы дают завышенные значения, поэтому прямое сравнение с цифрами из статей некорректно

In [21]:
final_model = EASE(reg=500).fit(train_val)
joblib.dump(final_model, '../src/models/EASE.pkl')

['../src/models/EASE.pkl']